# ICL Linear Classification Reproduction

In [22]:
import torch
import torch.nn as nn
import math
import numpy as np

## Data Generation

In [2]:
def data_gen(
    d: int,
    N: int,
    B: int,
    R: float,
    flip_prob: float = 0.0,
    device: str = "cpu",
    seed: int | None = None,
):

    if seed is not None:
        #keep base & flip RNG different. the original paper did not do this which led to issues when running their code.
        g_base = torch.Generator(device="cpu").manual_seed(seed)
        g_flip = torch.Generator(device="cpu").manual_seed(seed + 1)
    else:
        g_base = None
        g_flip = None

    mu = torch.randn(B, d, generator=g_base)#(B, d)
    mu = mu / mu.norm(dim=1, keepdim=True)  
    mu = R * mu                              

    labels = (torch.rand(B, N + 1, generator=g_base) > 0.5).float()#(B, N+1)
    y_signal = 2 * labels - 1                                  

    noise = torch.randn(B, N + 1, d, generator=g_base)#(B, N+1, d)

    x = (y_signal.unsqueeze(-1) * mu.unsqueeze(1) + noise)#(B, N+1, d)

    #introduce noise to labels
    if flip_prob > 0.0:
        flip_mask = torch.rand(B, N + 1, generator=g_flip) < flip_prob
        labels = torch.where(flip_mask, 1.0 - labels, labels)

    #output
    x_context = (x[:, :N, :]).to(device)           
    x_target = (x[:, -1, :]).to(device)           
    y_context = (labels[:, :N]).to(device)         
    y_target = (labels[:, -1]).to(device)        

    return (x_context, y_context, x_target, y_target)



### Data Testing

In [7]:
#Simple Testing
d, N, B, R = 1000, 20, 500, 5.0

x_ctx, y_ctx, x_tgt, y_tgt = data_gen(d, N, B, R, flip_prob=0.0, seed=100)

print("x_ctx shape:", x_ctx.shape)#(B, N, d)
print("y_ctx shape:", y_ctx.shape)#(B, N)
print("x_tgt shape:", x_tgt.shape)#(B, d)
print("y_tgt shape:", y_tgt.shape)#(B,)
#Check that printed shapes match comments

print("Context label mean:", y_ctx.mean().item())
print("Target label mean:", y_tgt.float().mean().item())
#Should be around 0.5, not exact due to RNG

x_ctx shape: torch.Size([500, 20, 1000])
y_ctx shape: torch.Size([500, 20])
x_tgt shape: torch.Size([500, 1000])
y_tgt shape: torch.Size([500])
Context label mean: 0.49160000681877136
Target label mean: 0.5139999985694885


In [9]:
#Label flip/noise testing
p = 0.3
seed = 100

#No flips
x_ctx_clean, y_ctx_clean, x_tgt_clean, y_tgt_clean = data_gen(
    d, N, B, R, flip_prob=0.0, seed=seed
)

#Some flips (p=0.3)
x_ctx_noisy, y_ctx_noisy, x_tgt_noisy, y_tgt_noisy = data_gen(
    d, N, B, R, flip_prob=p, seed=seed
)

#Should return True for both since features should not be affected
print("Context X equal:", torch.allclose(x_ctx_clean, x_ctx_noisy))
print("Target X equal:", torch.allclose(x_tgt_clean, x_tgt_noisy))

#Should return near 0.3 for both as p = 0.3
ctx_flip_rate = (y_ctx_clean != y_ctx_noisy).float().mean().item()
tgt_flip_rate = (y_tgt_clean != y_tgt_noisy).float().mean().item()

print(f"Context flip rate ~ {ctx_flip_rate:.3f} (target {p})")
print(f"Target  flip rate ~ {tgt_flip_rate:.3f} (target {p})")

Context X equal: True
Target X equal: True
Context flip rate ~ 0.297 (target 0.3)
Target  flip rate ~ 0.332 (target 0.3)


Now that all testing is done and if output looks correct as specificed by in-line comments, we can move on to model architecture.

## Model Definition

In [14]:
class LinearClassifier(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        
        #Linear layer
        self.W = nn.Linear(d, d, bias=False)
        nn.init.zeros_(self.W.weight)

    #only forward pass
    def forward(self, x_ctx, y_ctx, x_tgt):
        B, N, d = x_ctx.shape

        y_signal = 2*y_ctx - 1 
        
        weighted = y_signal.unsqueeze(-1) * x_ctx   
        mu_hat = weighted.mean(dim=1)            

        v = self.W(mu_hat)     

        return (v * x_tgt).sum(dim=1)
    
    #use same model as target prediction to predict in context labels
    def compute_in_context_preds(self, x_ctx, y_ctx):

        B, N, d = x_ctx.shape
        y_signal = 2*y_ctx - 1

        mu_hat = (y_signal.unsqueeze(-1) * x_ctx).mean(dim=1)  
        v = self.W(mu_hat)             
        logits = (v.unsqueeze(1) * x_ctx).sum(dim=2)           
        
        return (logits > 0).float()



### Simple Model Tests

In [7]:
#test output shape after forward pass
def test_forward_shapes():
    d = 100
    N = 5
    B = 10

    x_ctx = torch.randn(B, N, d)
    y_ctx = (torch.rand(B, N) > 0.5).float()
    x_tgt = torch.randn(B, d)

    model = LinearClassifier(d)

    logits = model(x_ctx, y_ctx, x_tgt)

    print("logits shape:", logits.shape)
    assert logits.shape == (B,)

test_forward_shapes()

logits shape: torch.Size([10])


In [9]:
#forward pass is working and gradients exist 

def test_pass_and_grad():
    d = 50
    N = 3
    B = 4

    x_ctx = torch.randn(B, N, d, requires_grad=True)
    y_ctx = (torch.rand(B, N) > 0.5).float()
    x_tgt = torch.randn(B, d, requires_grad=True)

    model = LinearClassifier(d)

    logits = model(x_ctx, y_ctx, x_tgt).sum()
    logits.backward()

    assert model.W.weight.grad is not None
    print("Gradient shape:", model.W.weight.grad.shape)
    
test_pass_and_grad()

Gradient shape: torch.Size([50, 50])


Now that we have ran some simple sanity tests on our model architecture, we can move on to creating the training loop.

## Training Loop

In [28]:
#Helper Methods

#gets in context predictions + accuracy on a validation batch
def evaluate(model, d, N, B_val, R_val, flip_val=0.0, device="cpu"):
    model.eval()
    with torch.no_grad():

        #sample batch
        x_ctx, y_ctx, x_tgt, y_tgt = data_gen(d, N, B_val, R_val, flip_prob = flip_val, device=device)

        #forward pass
        logits = model(x_ctx, y_ctx, x_tgt)
        val_loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_tgt.float())

        #compute target acc
        preds = (logits > 0).float()
        val_acc = (preds == y_tgt).float().mean().item()

        #compute icl acc
        ctx_preds = model.compute_in_context_preds(x_ctx, y_ctx)
        ctx_acc = (ctx_preds == y_ctx).float().mean().item()

        return val_loss.item(), val_acc, ctx_acc


In [29]:
def train_model(
    model,
    d: int,
    N: int,
    B: int,
    R_train: float,
    R_val: float,
    flip_train: float = 0.0,
    flip_val: float = 0.0,
    steps: int = 300,
    lr: float = 1e-2,
    device: str = "cpu"
):
    model = model.to(device)
    optim = torch.optim.SGD(model.parameters(), lr=lr)

    for step in range(steps):

        # --- Training batch ---
        x_ctx, y_ctx, x_tgt, y_tgt = data_gen(
            d, N, B, R_train, flip_prob=flip_train, device=device
        )

        logits = model(x_ctx, y_ctx, x_tgt)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, y_tgt.float()
        )

        # update
        optim.zero_grad()
        loss.backward()
        optim.step()

        # training accuracy
        train_acc = ((logits > 0).float() == y_tgt).float().mean().item()

        # --- Validation & in-context accuracy ---
        val_loss, val_acc, ctx_acc = evaluate(
            model, d, N, B, R_val, flip_val=flip_val, device=device
        )

        if step % 20 == 0:
            print(
                f"Step {step:03d} | "
                f"Train Loss: {loss.item():.4f} | Train Acc: {train_acc:.3f} | "
                f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f} | "
                f"In-Context Acc: {ctx_acc:.3f}"
            )

    return model


In [16]:
def test_params_change():
    d = 50
    model = LinearClassifier(d)

    # copy weights
    before = model.W.weight.detach().clone()

    # tiny train step
    x_ctx, y_ctx, x_tgt, y_tgt = data_gen(d, 5, 4, 5.0)
    logits = model(x_ctx, y_ctx, x_tgt)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_tgt.float())
    loss.backward()

    with torch.no_grad():
        model.W.weight -= 0.1 * model.W.weight.grad

    after = model.W.weight.detach().clone()

    print("Did parameters change?", not torch.allclose(before, after))
test_params_change()

Did parameters change? True


## Running the Training Loop

In [31]:
d = 1000
N = 20
B = 64
R = d**0.3

model = LinearClassifier(d=d)
train_model(
    model,
    d,
    N,
    B,
    R,
    R,
    flip_train=0.0,
    flip_val=0.2,
    steps=300,
    device="cpu",
)


Step 000 | Train Loss: 0.6931 | Train Acc: 0.531 | Val Loss: 0.6848 | Val Acc: 0.641 | In-Context Acc: 0.627
Step 020 | Train Loss: 0.5574 | Train Acc: 0.812 | Val Loss: 0.6616 | Val Acc: 0.625 | In-Context Acc: 0.919
Step 040 | Train Loss: 0.4231 | Train Acc: 1.000 | Val Loss: 0.5534 | Val Acc: 0.797 | In-Context Acc: 0.948
Step 060 | Train Loss: 0.3710 | Train Acc: 0.953 | Val Loss: 0.5613 | Val Acc: 0.812 | In-Context Acc: 0.946
Step 080 | Train Loss: 0.3021 | Train Acc: 0.984 | Val Loss: 0.5308 | Val Acc: 0.828 | In-Context Acc: 0.951
Step 100 | Train Loss: 0.2586 | Train Acc: 1.000 | Val Loss: 0.4957 | Val Acc: 0.844 | In-Context Acc: 0.960
Step 120 | Train Loss: 0.2298 | Train Acc: 0.984 | Val Loss: 0.5720 | Val Acc: 0.766 | In-Context Acc: 0.959
Step 140 | Train Loss: 0.1825 | Train Acc: 1.000 | Val Loss: 0.6483 | Val Acc: 0.734 | In-Context Acc: 0.962
Step 160 | Train Loss: 0.1893 | Train Acc: 0.984 | Val Loss: 0.4511 | Val Acc: 0.859 | In-Context Acc: 0.955
Step 180 | Train Lo

LinearClassifier(
  (W): Linear(in_features=1000, out_features=1000, bias=False)
)